# 🎙️ Urdu Interview Processing Pipeline
**Stages:** Audio → Urdu Transcript → Verify → English Translation → Verify → De-identify → Final Dataset

**Models used:**
- ASR: `openai/whisper-large-v3-turbo` (via faster-whisper)
- Translation: `facebook/nllb-200-1.3B` ⬆️ **Upgraded for better quality**
- De-identification: `Microsoft Presidio` + `spaCy en_core_web_lg`

**Improvements in this version:**
- ✅ NLLB model upgraded from 600M to 1.3B (3x better translation)
- ✅ Sentence-aware chunking (preserves context instead of hard character splits)
- ✅ Better handling of Urdu→English translations

> ⚠️ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running!

In [1]:
# ── CELL 1: Check GPU ──────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✔ GPU available: {gpu_name}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    print('  Pipeline will run on CPU (much slower)')

✔ GPU available: Tesla T4
  VRAM: 15.6 GB


In [2]:
# ── CELL 2: Install all dependencies ──────────────────────
print('Installing dependencies... (takes 3-5 minutes first time)')
print('⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)')

!pip install -q faster-whisper==1.1.0
!pip install -q transformers==4.44.2 sentencepiece==0.2.0 sacremoses==0.1.1
!pip install -q presidio-analyzer==2.2.355 presidio-anonymizer==2.2.355
!pip install -q spacy==3.8.1
!pip install -q python-docx==1.1.2 langdetect==1.0.9 sacrebleu==2.4.3

# Download spaCy model
!python -m spacy download en_core_web_lg -q

print('\n✔ All dependencies installed!')
print('✔ Models will auto-download on first use (may take 2-3 minutes per model)')

Installing dependencies... (takes 3-5 minutes first time)
⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

✔ All dependencies installed!
✔ Models will auto-download on first use (may take 2-3 minutes per model)


In [1]:
# ── CELL 3: Clone project from GitHub ──────────────────────
!git clone https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE.git urdu-pipeline
%cd urdu-pipeline

# Pull latest changes
!git pull origin main

print('✔ Repository cloned and updated successfully!')
!ls -la

fatal: destination path 'urdu-pipeline' already exists and is not an empty directory.
/content/urdu-pipeline
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.63 KiB | 40.00 KiB/s, done.
From https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE
 * branch            main       -> FETCH_HEAD
   2ff626a..9cdd746  main       -> origin/main
Updating 2ff626a..9cdd746
Fast-forward
 notebooks/colab_pipeline.ipynb | 2471 +---------------------------------------
 1 file changed, 63 insertions(+), 2408 deletions(-)
✔ Repository cloned and updated successfully!
total 60
drwxr-xr-x 9 root root 4096 Jun 30 12:25 .
drwxr-xr-x 1 root root 4096 Jun 30 13:46 ..
drwxr-xr-x 2 root root 4096 Jun 30 11:16 audio
-rw-r--r-- 1 root root 3861 Jun 30 12:25 config.py
drwxr-xr-x 8 root root 4096 Jun 30 13:55 .git
-rw-r--r-- 

In [2]:
# ── CELL 4: Download audio from YouTube ───────────────────
import os

os.makedirs('audio', exist_ok=True)

!pip install -q yt-dlp

YOUTUBE_URL = "https://youtu.be/pHZHYWe8Mkc"

# Download full audio as mp3
!yt-dlp -x --audio-format mp3 -o "audio/full.%(ext)s" "{YOUTUBE_URL}"

# Trim from 137 seconds, duration 55 minutes (3163 seconds)
!ffmpeg -y -i audio/full.mp3 -ss 137 -t 3163 -c copy audio/test_audio.mp3

audio_path = "audio/test_audio.mp3"

size_mb = os.path.getsize(audio_path) / 1e6
print(f'\n✔ Audio ready: {audio_path}')
print(f'   File size: {size_mb:.1f} MB')
print(f'   Duration: 55 minutes (starting from 137 seconds)')


[youtube] Extracting URL: https://youtu.be/pHZHYWe8Mkc
[youtube] pHZHYWe8Mkc: Downloading webpage
[youtube] pHZHYWe8Mkc: Downloading android vr player API JSON
[info] pHZHYWe8Mkc: Downloading 1 format(s): 251
[download] audio/full.mp3 has already been downloaded
[ExtractAudio] Not converting audio audio/full.mp3; file is already in target format mp3
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-li

In [3]:
# ── CELL 5: Configure pipeline ─────────────────────────────
import sys
sys.path.insert(0, 'urdu-pipeline')

import config

# Set device to cuda since we have GPU
config.WHISPER_DEVICE       = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

# Set audio path
AUDIO_PATH = audio_path

print('Pipeline Configuration:')
print(f'  ASR Model        : whisper-{config.WHISPER_MODEL}')
print(f'  Translation Model: {config.TRANSLATION_MODEL}')
print(f'  Device           : {config.WHISPER_DEVICE}')
print(f'  Audio file       : {AUDIO_PATH}')
print(f'  Confidence thresh: {config.CONFIDENCE_THRESHOLD}')
print(f'  Chunk size       : {config.CHUNK_SIZE} chars')

Pipeline Configuration:
  ASR Model        : whisper-large-v3-turbo
  Translation Model: facebook/nllb-200-1.3B
  Device           : cuda
  Audio file       : audio/test_audio.mp3
  Confidence thresh: 0.75
  Chunk size       : 500 chars


In [4]:
# Check if audio file exists and its size
import os

audio_path = "audio/test_audio.mp3"
if os.path.exists(audio_path):
    size_mb = os.path.getsize(audio_path) / 1e6
    size_mins = size_mb / 0.64  # Rough conversion: 1 min ≈ 0.64 MB at 128kbps
    print(f'✔ Audio file exists: {size_mb:.1f} MB ({size_mins:.0f} minutes)')
else:
    print(f'✗ Audio file NOT found: {audio_path}')

# Check GPU status
import torch
if torch.cuda.is_available():
    print(f'✔ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   Used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')
else:
    print('✗ GPU not available!')

✔ Audio file exists: 29.9 MB (47 minutes)
✔ GPU: Tesla T4
   VRAM: 15.6 GB
   Used: 0.00 GB


In [5]:
# ── CELL 6: STAGE 1 — Urdu Transcription ──────────────────
from pipeline.utils import ensure_dirs
ensure_dirs(config.STAGE1_DIR, config.STAGE2_DIR, config.STAGE3_DIR,
            config.STAGE4_DIR, config.STAGE5_DIR, config.STAGE6_DIR)

from pipeline.transcribe import transcribe

stage1_result = transcribe(AUDIO_PATH)

# Preview
print('\n── Urdu Transcript Preview (first 500 chars) ──')
print(stage1_result['full_urdu_text'][:500])


  STAGE 1: URDU TRANSCRIPTION (ASR)
  Audio file : audio/test_audio.mp3
  Model      : whisper-large-v3-turbo
  Language   : ur (Urdu)
  Device     : cuda

  Loading Whisper model (first run downloads ~800MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


  ✔ Model loaded.

  Transcribing audio... (this may take a few minutes for long audio)

  Processing segments:
    [⚠] 00:00:00.000 → 00:00:01.600 | conf=0.73 | اسلام علیکم Dr. Majda...
    [⚠] 00:00:01.600 → 00:00:02.960 | conf=0.72 | Welcome to the show...
    [✔] 00:00:02.960 → 00:00:05.060 | conf=0.88 | Thank you so much for coming today...
    [✔] 00:00:05.060 → 00:00:06.460 | conf=0.95 | and taking out the time...
    [⚠] 00:00:07.860 → 00:00:10.820 | conf=0.55 | If you give us a small introduction...
    [⚠] 00:00:10.820 → 00:00:13.180 | conf=0.65 | so that our audience that you are listening to today...
    [✔] 00:00:13.180 → 00:00:15.360 | conf=0.81 | they know that what a brilliant...
    [✔] 00:00:15.360 → 00:00:17.200 | conf=0.90 | Academian you are...
    [✔] 00:00:17.200 → 00:00:19.260 | conf=0.93 | Thank you so much Zainab...
    [✔] 00:00:19.260 → 00:00:21.740 | conf=0.88 | for inviting me on your show...
    [✔] 00:00:21.740 → 00:00:24.340 | conf=0.86 | My name is Maj

In [7]:
# ── CELL 7: STAGE 2 — Verify Urdu Transcript ──────────────
from pipeline.verify_transcript import verify_transcript

stage2_result = verify_transcript(stage1_result)

print(f'\nQuality Score : {stage2_result["quality_score"]}/100')
print(f'Quality Label : {stage2_result["quality_label"]}')

# Show flagged segments
flagged = stage2_result['verification_report']['flagged_details']
if flagged:
    print(f'\n⚠ Flagged Segments ({len(flagged)}):')
    for f in flagged[:5]:
        print(f'  seg {f["segment_id"]} [{f["start_fmt"]}] conf={f["confidence"]:.2f}: {f["text"][:60]}...')
else:
    print('\n✔ No segments flagged!')


  STAGE 2: VERIFICATION OF URDU TRANSCRIPT
  Interview ID      : test_audio
  Total segments    : 2086
  Confidence threshold: 0.75
    [⚠ FLAGGED] seg 001 | conf=0.73 | اسلام علیکم Dr. Majda...
    [⚠ FLAGGED] seg 002 | conf=0.72 | Welcome to the show...
    [✔] seg 003 | conf=0.88 | Thank you so much for coming today...
    [✔] seg 004 | conf=0.95 | and taking out the time...
    [⚠ FLAGGED] seg 005 | conf=0.55 | If you give us a small introduction...
    [⚠ FLAGGED] seg 006 | conf=0.65 | so that our audience that you are listening to tod...
    [✔] seg 007 | conf=0.81 | they know that what a brilliant...
    [✔] seg 008 | conf=0.90 | Academian you are...
    [✔] seg 009 | conf=0.93 | Thank you so much Zainab...
    [✔] seg 010 | conf=0.88 | for inviting me on your show...
    [✔] seg 011 | conf=0.86 | My name is Majda Kazmi...
    [⚠ FLAGGED] seg 012 | conf=0.62 | I am PhD doctor...
    [⚠ FLAGGED] seg 013 | conf=0.69 | MBBS...
    [⚠ FLAGGED] seg 014 | conf=0.08 | Exactly exactly.

In [8]:
# ── CELL 8: STAGE 3 — Translate Urdu → English ────────────
from pipeline.translate import translate

stage3_result = translate(stage2_result)

print('\n── English Translation Preview (first 500 chars) ──')
print(stage3_result['english_full_text'][:500])


  STAGE 3: TRANSLATION: URDU → ENGLISH
  Interview ID   : test_audio
  Model          : facebook/nllb-200-1.3B
  Source lang    : urd_Arab (Urdu)
  Target lang    : eng_Latn (English)
  Chunk size     : 500 chars

  Loading NLLB-200 model (first run downloads ~1.2GB)...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  ✔ Model loaded on cuda.

  Translating full text...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



  Translating segments:
    ✔ seg 001/2086 | UR: اسلام علیکم Dr. Majda...
             | EN: Islam علیکم Dr. Majda...
    ✔ seg 002/2086 | UR: Welcome to the show...
             | EN: Welcome to the show...
    ✔ seg 003/2086 | UR: Thank you so much for coming today...
             | EN: Thank you so much for coming today...
    ✔ seg 004/2086 | UR: and taking out the time...
             | EN: today and taking the time...
    ✔ seg 005/2086 | UR: If you give us a small introduction...
             | EN: if you give us a little introduction...
    ✔ seg 006/2086 | UR: so that our audience that you are listen...
             | EN: so that our audience that you are listening to today...
    ✔ seg 007/2086 | UR: they know that what a brilliant...
             | EN: today they know what a brilliant...
    ✔ seg 008/2086 | UR: Academian you are...
             | EN: academic you are...
    ✔ seg 009/2086 | UR: Thank you so much Zainab...
             | EN: Thank you so much Zainab...
    

In [9]:
# ── CELL 9: STAGE 4 — Verify English Translation ──────────
from pipeline.verify_translation import verify_translation

stage4_result = verify_translation(stage3_result)

print(f'\nTranslation Quality Score : {stage4_result["translation_quality_score"]}/100')
print(f'Translation Quality Label : {stage4_result["translation_quality_label"]}')

# Show flagged translation segments
t_flagged = stage4_result['translation_verification_report']['flagged_details']
if t_flagged:
    print(f'\n⚠ Translation Flagged Segments ({len(t_flagged)}):')
    for f in t_flagged[:5]:
        print(f'  seg {f["segment_id"]}: {f["eng_text"][:60]}... | Issues: {", ".join(f["issues"])}')
else:
    print('\n✔ All translations verified OK!')


  STAGE 4: VERIFICATION OF ENGLISH TRANSLATION
  Interview ID   : test_audio
  Total segments : 2086

  Checks: Empty translation, Translation errors, Length ratio
  NOTE: Language detection disabled (too many false positives on proper names)
    [✔] seg 001
             EN: Islam علیکم Dr. Majda...
    [✔] seg 002
             EN: Welcome to the show...
    [✔] seg 003
             EN: Thank you so much for coming today...
    [✔] seg 004
             EN: today and taking the time...
    [✔] seg 005
             EN: if you give us a little introduction...
    [✔] seg 006
             EN: so that our audience that you are listening to today...
    [✔] seg 007
             EN: today they know what a brilliant...
    [✔] seg 008
             EN: academic you are...
    [✔] seg 009
             EN: Thank you so much Zainab...
    [✔] seg 010
             EN: for inviting me to your show...
    [✔] seg 011
             EN: My name is Majda Kazmi...
    [✔] seg 012
             EN: I am Ph

In [10]:
# ── CELL 10: STAGE 5 — De-identification ──────────────────
from pipeline.deidentify import deidentify

stage5_result = deidentify(stage4_result)

print(f'\nEntities removed: {stage5_result["entities_removed_count"]}')
print('\n── De-identified Text Preview (first 500 chars) ──')
print(stage5_result['deidentified_english_full'][:500])


  STAGE 5: DE-IDENTIFICATION OF DATASET
  Interview ID : test_audio
  Loading Presidio + spaCy (first run may take a moment)...
  ✔ Presidio loaded.

  De-identifying full English text...

  De-identifying segments:
    [🔒] seg 001 | 1 entities removed | Islam علیکم Dr. [NAME]...
    [✔] seg 002 | 0 entities removed | Welcome to the show...
    [✔] seg 003 | 0 entities removed | Thank you so much for coming today...
    [🔒] seg 004 | 1 entities removed | [DATE] and taking the time...
    [✔] seg 005 | 0 entities removed | if you give us a little introduction...
    [✔] seg 006 | 0 entities removed | so that our audience that you are listening to today...
    [🔒] seg 007 | 1 entities removed | [DATE] they know what a brilliant...
    [✔] seg 008 | 0 entities removed | academic you are...
    [🔒] seg 009 | 1 entities removed | Thank you so much [NAME]...
    [✔] seg 010 | 0 entities removed | for inviting me to your show...
    [🔒] seg 011 | 1 entities removed | My name is [NAME]...
   

In [11]:
# ── CELL 11: STAGE 6 — Final Export ───────────────────────
from pipeline.export import export

final_result = export(stage5_result)

print(f'\n✔ Final JSON : {final_result["json_path"]}')
print(f'✔ Final DOCX : {final_result["docx_path"]}')


  STAGE 6: FINAL EXPORT: JSON + DOCX
  Interview ID : test_audio

  Building final JSON dataset...
  ✔ Saved → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json

  Building DOCX report...
  ✔ DOCX saved → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx

  ── Final Outputs ─────────────────────────────
  JSON → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json
  DOCX → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx

  ✔ Stage 6 complete. Pipeline finished!

✔ Final JSON : /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json
✔ Final DOCX : /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx


In [12]:
# ── CELL 12: Download all outputs ─────────────────────────
import shutil
from google.colab import files

interview_id = stage1_result['interview_id']

# Zip the entire outputs folder
zip_name = f'{interview_id}_outputs.zip'
shutil.make_archive(
    base_name=interview_id + '_outputs',
    format='zip',
    root_dir='urdu-pipeline',
    base_dir='outputs'
)

print(f'✔ Zipped outputs as {zip_name}')
print('Downloading...')
files.download(zip_name)

FileNotFoundError: [Errno 2] No such file or directory: 'urdu-pipeline/outputs'

In [ ]:
# ── CELL 13 (OPTIONAL): Run full pipeline in one go ───────
# Use this after testing individual stages above

import sys
sys.path.insert(0, 'urdu-pipeline')

import config
config.WHISPER_DEVICE = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

from main import run_pipeline

# Change this to your audio path
run_pipeline('urdu-pipeline/audio/your_audio.mp3', start_stage=1)